This notebook checks the accuracy of the LLama 3.1 8b model, in the context of plain statements with an evaluation instruction to retreive the true/false judgement. If the accuracy is low than this will be a basis to change the model before starting with the CoT implementation, and moving to a higher parameter/newer/better model that is capable of getting near full accuracy on all tasks.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import re
import torch
from tqdm import tqdm
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

### Testing the Accuracy of LLama 3.1 8b

### Loading the Model

In [2]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", device_map="cuda")
print(model.device)
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

cuda:0


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."<|eot_id|>


### Loading the Datasets

In [3]:
# Load in the Factual Datasets
F0_train, F0_test = pd.read_csv("../dataset/F0_train.csv")[["statement", "label"]], pd.read_csv("../dataset/F0_test.csv")[["statement", "label"]]
F1_train, F1_test = pd.read_csv("../dataset/F1_train.csv")[["statement", "label"]], pd.read_csv("../dataset/F1_test.csv")[["statement", "label"]]
F2_train, F2_test = pd.read_csv("../dataset/F2_train.csv"), pd.read_csv("../dataset/F2_test.csv")
F3_train, F3_test = pd.read_csv("../dataset/F3_train.csv"), pd.read_csv("../dataset/F3_test.csv")
F4_train, F4_test = pd.read_csv("../dataset/F4_train.csv"), pd.read_csv("../dataset/F4_test.csv")
F5_train, F5_test = pd.read_csv("../dataset/F5_train.csv"), pd.read_csv("../dataset/F5_test.csv")

# Load in the Arithmatic Statements
A1_train, A1_test = pd.read_csv("../dataset/A1_train.csv"), pd.read_csv("../dataset/A1_test.csv")
A2_train, A2_test = pd.read_csv("../dataset/A2_train.csv"), pd.read_csv("../dataset/A2_test.csv")
A3_train, A3_test = pd.read_csv("../dataset/A3_train.csv"), pd.read_csv("../dataset/A3_test.csv")


In [4]:
datasets = {
    "F0_train": F0_train, "F0_test": F0_test,
    "F1_train": F1_train, "F1_test": F1_test,
    "F2_train": F2_train, "F2_test": F2_test,
    "F3_train": F3_train, "F3_test": F3_test,
    "F4_train": F4_train, "F4_test": F4_test,
    "F5_train": F5_train, "F5_test": F5_test,
    "A1_train": A1_train, "A1_test": A1_test,
    "A2_train": A2_train, "A2_test": A2_test,
    "A3_train": A3_train, "A3_test": A3_test,
}

for name, df in datasets.items():
    print(f"{name}: {len(df)}")


F0_train: 1194
F0_test: 512
F1_train: 1194
F1_test: 512
F2_train: 1194
F2_test: 512
F3_train: 1398
F3_test: 600
F4_train: 1394
F4_test: 598
F5_train: 1383
F5_test: 593
A1_train: 700
A1_test: 300
A2_train: 700
A2_test: 300
A3_train: 700
A3_test: 300


### Defining Generation and Extraction Funcitons

In [5]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

EVAL_INSTRUCTIONS = (
    "Evaluate the statement for whether it is true or false. "
    "write exactly one line and nothing after it: "
    "\nAnswer: Yes or Answer: No"
)

# Primary: the requested "Answer: Yes/No" format.
# Fallback 1: model used True/False instead of Yes/No.
# Fallback 2: model skipped the "Answer:" prefix and just wrote the bare word,
# possibly on its own line — check the LAST non-empty line specifically, since
# an earlier line mentioning "yes"/"no" as part of normal text shouldn't count.
ANSWER_RE = re.compile(r"Answer:\s*(Yes|No|True|False)", re.IGNORECASE)
BARE_RE = re.compile(r"^(Yes|No|True|False)\.?$", re.IGNORECASE)
LEADING_RE = re.compile(r"^(Yes|No|True|False)\b", re.IGNORECASE)  # verdict word first, explanation may follow

def extract_answer(model_output):
    match = ANSWER_RE.search(model_output)
    if match is None:
        lines = [l.strip() for l in model_output.strip().splitlines() if l.strip()]
        if lines:
            match = BARE_RE.match(lines[-1]) or LEADING_RE.match(lines[-1])
    if match is None:
        print(f"Error: {model_output!r}")
        return None
    word = match.group(1).strip().lower()
    return word in ("yes", "true")

def generate_output(model, statements, with_chat_template=True, batch_size=16, stop_strings=["Answer: Yes", "Answer: No"]):
    statements = list(statements)
    outputs = []

    for i in tqdm(range(0, len(statements), batch_size)):
        # Empty the GPU cache every 5 batch
        if i % 5 == 0:
            torch.cuda.empty_cache()
        statements_temp = [f"{s}\n\n{EVAL_INSTRUCTIONS}" for s in statements[i: i+batch_size]]
        
        # Tokenize the statments
        if with_chat_template:
            messages = [[{"role": "user", "content": s}] for s in statements_temp]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to(model.device)
        else:
            inputs = tokenizer(statements_temp, return_tensors="pt", padding=True).to(model.device)

        # Define the generation key-word arguments
        kwargs = {**inputs, "max_new_tokens": 1000, "do_sample": False}
        if stop_strings is not None:
            kwargs["stop_strings"] = stop_strings
            kwargs["tokenizer"] = tokenizer

        # Generate output and decode
        with torch.no_grad():
            output_ids = model.generate(**kwargs)
        gen_only = output_ids[:, inputs["input_ids"].shape[-1]:]  # drop the prompt, keep only new tokens
        texts = tokenizer.batch_decode(gen_only, skip_special_tokens=True)
        outputs.extend(texts)

    return outputs


In [6]:
task_datasets = {
    "F0": pd.concat([F0_train, F0_test], ignore_index=True),
    "F1": pd.concat([F1_train, F1_test], ignore_index=True),
    "F2": pd.concat([F2_train, F2_test], ignore_index=True),
    "F3": pd.concat([F3_train, F3_test], ignore_index=True),
    "F4": pd.concat([F4_train, F4_test], ignore_index=True),
    "F5": pd.concat([F5_train, F5_test], ignore_index=True),
    "A1": pd.concat([A1_train, A1_test], ignore_index=True),
    "A2": pd.concat([A2_train, A2_test], ignore_index=True),
    "A3": pd.concat([A3_train, A3_test], ignore_index=True),
}

In [7]:
for task, dataset in tqdm(task_datasets.items()):
    output_texts = generate_output(model, dataset["statement"], with_chat_template=True, batch_size=16)
    correct = 0
    for model_output, ground_truth in zip(output_texts, dataset["label"]):
        if extract_answer(model_output) == ground_truth:
            correct += 1
    print(f"Accuracy Percentage for {task}: {correct * 100/len(output_texts)}%")

 11%|█         | 1/9 [01:54<15:13, 114.13s/it]

Accuracy Percentage for F0: 97.42086752637749%


 22%|██▏       | 2/9 [04:26<15:57, 136.83s/it]

Accuracy Percentage for F1: 93.96248534583822%


 33%|███▎      | 3/9 [07:13<15:01, 150.28s/it]

...

Accuracy Percentage for A1: 51.6%


 89%|████████▉ | 8/9 [18:59<01:58, 118.48s/it]

Accuracy Percentage for A2: 50.0%


100%|██████████| 9/9 [20:09<00:00, 134.43s/it]

Accuracy Percentage for A3: 50.1%


### Testing the Accuracy of DeepSeek-R1-Distill-Llama-8B

In [5]:
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B", device_map="cuda", dtype=torch.bfloat16)
print(model.device)
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

cuda:0
Greetings! I'm DeepSeek-R1, an artificial intelligence assistant created by DeepSeek. I'm at your service and would be delighted to assist you with any inquiries or tasks you may have.
</think>

Greetings! I'm DeepSeek-R1, an artificial intelligence assistant created by DeepSeek. I'm at your service and would be delighted to assist you with any inquiries or tasks you may have.<｜end▁of▁sentence｜>


In [ ]:
# Checking the accuracy score for F5
output_texts = generate_output(model, task_datasets["F5"]["statement"], with_chat_template=True, batch_size=8, stop_strings=None)
correct = 0
for model_output, ground_truth in zip(output_texts, task_datasets["F5"]["label"]):
    if extract_answer(model_output) == ground_truth:
        correct += 1
print(f"Accuracy Percentage for F5: {correct * 100/len(output_texts)}%")

100%|██████████| 247/247 [2:30:46<00:00, 36.63s/it]  

Error: "Okay, so I need to figure out if the statement about the cities is true or false. The statement says that exactly 2 of the given cities are in South Africa and 1 in Uganda. The list provided is Bloemfontein, Durban, Zhabei, Benoni, Quito, Evaton.\n\nFirst, I should identify which countries each city is in. Let me go through them one by one.\n\nBloemfontein: I think this is a city in South Africa. I've heard it's one of the major cities there.\n\nDurban: Definitely in South Africa as well. It's a coastal city and a significant place in SA.\n\nZhabei: Hmm, not sure about this one. I believe it might be in China because I recall a city named Zhabei, possibly near Shanghai. But I'm not entirely certain, maybe I should double-check later.\n\nBenoni: This sounds like it could be in South Africa too. I think it's a smaller city but still within SA.\n\nQuito: That definitely rings a bell as being in Ecuador. So that's in South America, not Africa.\n\nEvaton: I'm pretty sure this is in 

In [ ]:
# Examples from A3 (Initial Testing)
output_texts = generate_output(model, task_datasets["A3"]["statement"][:5], with_chat_template=True, batch_size=8, stop_strings=None)
correct = 0
for model_output, ground_truth in zip(output_texts, task_datasets["A3"]["label"][:5]):
    print(model_output, "\n", ground_truth, "\n")
    if extract_answer(model_output) == ground_truth:
        correct += 1
print(f"Accuracy Percentage for A3 Examples: {correct * 100/len(output_texts)}%")

100%|██████████| 1/1 [00:14<00:00, 14.34s/it]

Okay, so I need to figure out if the equation (33 + 31) * (7 * 16) equals 7171. Let me break it down step by step.

First, I'll handle the addition inside the first parentheses: 33 + 31. Adding those together, 33 plus 31 is 64. So now the equation simplifies to 64 multiplied by (7 * 16).

Next, I'll compute the multiplication inside the second parentheses: 7 times 16. I know that 7 times 10 is 70, and 7 times 6 is 42, so adding those together gives me 70 + 42, which is 112. So now the equation is 64 multiplied by 112.

Now, I need to calculate 64 times 112. I can break this down to make it easier. Let's see, 64 times 100 is 6400, and 64 times 12 is 768. Adding those two results together: 6400 + 768. Hmm, 6400 plus 700 is 7100, and then plus 68 is 7168. Wait, that's 7168, but the original statement says it's 7171. That's a problem because 7168 isn't equal to 7171. Did I make a mistake somewhere?

Let me double-check my calculations. Maybe I messed up the multiplication part. So, 7 times

In [9]:
# Checking the full accuracy score for A3
output_texts = generate_output(model, task_datasets["A3"]["statement"], with_chat_template=True, batch_size=8, stop_strings=None)
correct = 0
for model_output, ground_truth in zip(output_texts, task_datasets["A3"]["label"]):
    if extract_answer(model_output) == ground_truth:
        correct += 1
print(f"Accuracy Percentage for A3: {correct * 100/len(output_texts)}%")

100%|██████████| 125/125 [1:01:01<00:00, 29.30s/it]

Error: "Okay, so I need to evaluate whether the statement (9 * 7) * (18 * 20) equals 22676. Let me break this down step by step to make sure I don't make any mistakes.\n\nFirst, I'll calculate each multiplication separately. Starting with 9 multiplied by 7. I know that 9 times 7 is 63. So, (9 * 7) equals 63.\n\nNext, I'll handle the second part, which is 18 multiplied by 20. I'm a bit unsure about this one, but I think 18 times 20 is 360. Let me double-check that. 18 times 20 is the same as 18 times 2 times 10, which is 36 times 10, so that's 360. Yep, that's correct.\n\nNow, the original expression is (63) multiplied by (360). I need to compute 63 times 360. Hmm, that might be a bit tricky. Let me think about how to approach this. Maybe I can break it down into smaller parts to make it easier.\n\nI know that 63 times 300 is 18,900 because 63 times 3 is 189, and then I add a zero for the hundreds place. Then, 63 times 60 is 3,780 because 63 times 6 is 378, and again, I add a zero for t

### Testing Accucary with a Front-tier Model (Non-Reasoning)

Now I will test the accuracy of a model from OpenAI API to see if there is a more fundamental issue with the task itself.

In [ ]:
# Finding and loading the OPENAI_API_KEY from a ".env" file / Replace with your file path
_ = load_dotenv(dotenv_path=r"llm_variables.env", override=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def get_completion(prompt, model="gpt-4.1-mini"):
    kwargs = {"model": model, "input": prompt}
    if not model.startswith(("o1", "o3", "o4")):
        kwargs["temperature"] = 0
    response = client.responses.create(**kwargs)
    return response.output_text

In [13]:
def generate_output_api(statements, model="gpt-4.1"):
    statements = list(statements)
    outputs = []

    for statement in statements:
        statement = f"{statement}\n\n{EVAL_INSTRUCTIONS}"

        model_output = get_completion(prompt=statement, model=model)
    
        outputs.append(model_output)

    return outputs

In [ ]:
outputs_F5_api = generate_output_api(F5_train["statement"][:100])
outputs_A3_api = generate_output_api(A3_train["statement"][:100])



for model_outputs, ground_truths, task in zip([outputs_F5_api, outputs_A3_api], [F5_train["label"][:100], A3_train["label"][:100]], ["F5", "A3"]):
    correct = 0
    for model_output, ground_truth in zip(model_outputs, ground_truths):
        if extract_answer(model_output) == ground_truth:
            correct += 1
    print(f"Accuracy Percentage for {task}: {correct * 100/len(model_outputs)}%")

Accuracy Percentage for F5: 72.8%
Accuracy Percentage for A3: 55.6%


### Testing Accucary with a Front-tier Model (Reasoning)

In [33]:
outputs_F5_api = generate_output_api(F5_train["statement"][:100], model="o3")
outputs_A3_api = generate_output_api(A3_train["statement"][:100], model="o3")

for model_outputs, ground_truths, task in zip([outputs_F5_api, outputs_A3_api], [F5_train["label"][:100], A3_train["label"][:100]], ["F5", "A3"]):
    correct = 0
    for model_output, ground_truth in zip(model_outputs, ground_truths):
        if extract_answer(model_output) == ground_truth:
            correct += 1
    print(f"Accuracy Percentage for {task}: {correct * 100/len(model_outputs)}%")

Accuracy Percentage for F5: 100.0%
Accuracy Percentage for A3: 100.0%


In [14]:
outputs_F5_api = generate_output_api(F5_train["statement"][:100], model="o3-mini")
outputs_A3_api = generate_output_api(A3_train["statement"][:100], model="o3-mini")

for model_outputs, ground_truths, task in zip([outputs_F5_api, outputs_A3_api], [F5_train["label"][:100], A3_train["label"][:100]], ["F5", "A3"]):
    correct = 0
    for model_output, ground_truth in zip(model_outputs, ground_truths):
        if extract_answer(model_output) == ground_truth:
            correct += 1
    print(f"Accuracy Percentage for {task}: {correct * 100/len(model_outputs)}%")

Accuracy Percentage for F5: 99.0%
Accuracy Percentage for A3: 100.0%
